# 🏥 VoiceAid Health — Smart Backend (Auto GPU → CPU Fallback)

This notebook **automatically detects whether a GPU is available** and configures all models accordingly.
No manual changes needed — just run both cells.

## How the Auto-Detection Works

At startup, the backend runs `torch.cuda.is_available()`. Based on the result:

| Setting | GPU Available ✅ | No GPU (CPU) ⚠️ |
|---------|-----------------|----------------|
| Device | `cuda` | `cpu` |
| Torch dtype | `float16` (fast) | `float32` (stable) |
| English ASR model | `whisper-medium` | `whisper-small` (faster) |
| Twi ASR model | fine-tuned whisper-small | fine-tuned whisper-small |
| Intent Predictor (LLM) | ✅ Qwen 2.5 1.5B enabled | ❌ Disabled (returns 501) |
| TTS | GPU-accelerated | CPU (slower) |
| Speed | 1–3s per chunk | 8–20s per chunk |

> ℹ️ Set **Runtime → Change Runtime Type → GPU (T4)** in Colab for best performance.
> If no GPU is allocated, the backend still works — just slower.

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q fastapi "uvicorn[standard]" transformers torch pydub soundfile \
    python-multipart scipy nest-asyncio websockets accelerate
!killall cloudflared 2>/dev/null || true
!rm -f cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O cloudflared
!chmod +x cloudflared

import torch
GPU_AVAILABLE = torch.cuda.is_available()
print(f'✅ Dependencies installed!')
print(f'🖥️  Mode: {"GPU (" + torch.cuda.get_device_name(0) + ")" if GPU_AVAILABLE else "CPU (no GPU detected)"}')
print(f'⚡  Intent Predictor (Qwen LLM): {"ENABLED" if GPU_AVAILABLE else "DISABLED — app will use local phrase chips"}')

## Cell 2 — Unified Smart Backend

### Section 1 — Device Auto-Detection
Detects GPU availability and sets `DEVICE`, `DTYPE`, and feature flags used throughout the app. All model loading functions read from these constants — no hardcoded `cuda` strings anywhere.

### Section 2 — App Setup & Middleware
Same WebSocket Origin Bypass (fixes 403 from Cloudflare) and CORS middleware.

### Section 3 — ASR Engine
Loads models onto whichever device is available. On CPU, switches English ASR to `whisper-small` for reasonable speed.

### Section 4 — ASR Routes
Batch POST and real-time WebSocket streaming — identical in both modes.

### Section 5 — Intent Predictor (LLM)
Qwen 2.5 1.5B only loads when GPU is available. On CPU, the endpoint returns `501` and the app automatically falls back to its built-in local medical phrase suggestions.

### Section 6 — TTS Engine
MMS VITS models adapt to whichever device is available.

### Section 7 — Deployment
Cloudflare Tunnel + Uvicorn. Copy the printed URL into `constants/config.ts`.

In [ ]:
import io, re, time, json, base64, torch, numpy as np, scipy.io.wavfile, subprocess
from fastapi import FastAPI, UploadFile, File, Form, WebSocket, WebSocketDisconnect
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from starlette.types import ASGIApp, Receive, Scope, Send
from pydantic import BaseModel
from transformers import (
    AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline,
    VitsModel, AutoTokenizer, AutoModelForCausalLM
)
import nest_asyncio, uvicorn

nest_asyncio.apply()

# ============================================================
# SECTION 1: Device Auto-Detection
# ============================================================
# All model loading uses DEVICE and DTYPE — no hardcoded 'cuda' strings.
# LLM_ENABLED controls whether the Qwen intent predictor is available.
# EN_ASR_MODEL is whisper-medium on GPU, whisper-small on CPU for speed.

GPU_AVAILABLE = torch.cuda.is_available()
DEVICE        = 'cuda' if GPU_AVAILABLE else 'cpu'
DTYPE         = torch.float16 if GPU_AVAILABLE else torch.float32
LLM_ENABLED   = GPU_AVAILABLE
EN_ASR_MODEL  = 'openai/whisper-medium' if GPU_AVAILABLE else 'openai/whisper-small'

print(f'\n🔧 Backend starting in {"GPU" if GPU_AVAILABLE else "CPU"} mode')
if GPU_AVAILABLE:
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'   Intent Predictor (LLM): {"ENABLED" if LLM_ENABLED else "DISABLED"}')

# ============================================================
# SECTION 2: App Setup & Middleware
# ============================================================

app = FastAPI(title='VoiceAid Health Backend', description=f'Running on: {DEVICE.upper()}')

class WebSocketOriginBypassMiddleware:
    """Fixes 403 Forbidden on WebSocket connections through Cloudflare tunnels."""
    def __init__(self, app: ASGIApp): self.app = app
    async def __call__(self, scope: Scope, receive: Receive, send: Send):
        if scope['type'] == 'websocket':
            headers = [(k, v) for k, v in scope.get('headers', []) if k.lower() != b'origin']
            headers.append((b'origin', b'http://localhost'))
            scope['headers'] = headers
        await self.app(scope, receive, send)

app.add_middleware(WebSocketOriginBypassMiddleware)
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True,
                   allow_methods=['*'], allow_headers=['*'])

# ============================================================
# SECTION 3: Health Routes
# ============================================================

@app.get('/')
async def root():
    return {'message': 'VoiceAid Health Backend is Online!', 'gpu': GPU_AVAILABLE,
            'mode': 'gpu' if GPU_AVAILABLE else 'cpu', 'llm': LLM_ENABLED}

@app.get('/health')
async def health():
    return {'status': 'healthy', 'gpu': GPU_AVAILABLE, 'llm': LLM_ENABLED}

# ============================================================
# SECTION 4: ASR Engine (Whisper — GPU or CPU)
# ============================================================

asr_pipes = {}

ASR_GENERATE_KWARGS = {
    'max_new_tokens': 128,
    'temperature': 0.0,
    'condition_on_prev_tokens': False,
    'no_speech_threshold': 0.6,
    'repetition_penalty': 1.3,   # Prevents looping on unclear/slow speech
}

def load_asr(language='tw'):
    model_id = (
        'dennis-9/whisper-small_Akan_finetuned_v2' if language in ['tw', 'twi', 'akan']
        else EN_ASR_MODEL
    )
    if model_id in asr_pipes: return model_id
    print(f'\n🔥 Loading ASR ({model_id}) on {DEVICE.upper()}...')
    model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, torch_dtype=DTYPE).to(DEVICE)
    processor = AutoProcessor.from_pretrained(model_id)
    asr_pipes[model_id] = pipeline(
        'automatic-speech-recognition', model=model,
        tokenizer=processor.tokenizer, feature_extractor=processor.feature_extractor,
        torch_dtype=DTYPE, device=DEVICE, chunk_length_s=30,
    )
    print(f'✅ ASR Loaded: {model_id}')
    return model_id

def dysarthric_filter(text: str) -> str:
    """Remove stutter/repetition artifacts common in speech-impaired speech."""
    text = re.sub(r'\b(.+?)(?:\s+\1\b)+', r'\1', text, flags=re.IGNORECASE)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    return text.strip()

# ============================================================
# SECTION 5: ASR Routes
# ============================================================

@app.post('/asr/transcribe')
async def transcribe(file: UploadFile = File(...), language: str = Form('tw')):
    from pydub import AudioSegment
    audio = AudioSegment.from_file(io.BytesIO(await file.read()))
    audio = audio.set_channels(1).set_frame_rate(16000)
    samples = np.array(audio.get_array_of_samples()).astype(np.float32) / 32768.0
    model_id = load_asr(language)
    result = asr_pipes[model_id](samples, generate_kwargs=ASR_GENERATE_KWARGS)
    return {'text': dysarthric_filter(result['text']), 'model': model_id, 'language': language}

@app.websocket('/asr/stream')
async def stream_transcription(websocket: WebSocket):
    await websocket.accept()
    print(f'✅ WebSocket connected ({DEVICE.upper()} mode)')
    try:
        while True:
            data = await websocket.receive_text()
            message  = json.loads(data)
            audio_b64 = message.get('audio')
            language  = message.get('language', 'tw')
            chunk_id  = message.get('chunk_id', 0)
            if not audio_b64: continue

            from pydub import AudioSegment
            audio = (AudioSegment.from_file(io.BytesIO(base64.b64decode(audio_b64)))
                     .set_channels(1).set_frame_rate(16000))
            samples = np.array(audio.get_array_of_samples()).astype(np.float32) / 32768.0

            model_id = load_asr(language)
            result   = asr_pipes[model_id](samples, generate_kwargs=ASR_GENERATE_KWARGS)
            clean    = dysarthric_filter(result['text'])

            if not clean or len(clean.strip()) < 2:
                print(f'[ASR] ⏭️ Skipping silent chunk {chunk_id}')
                continue

            await websocket.send_json({
                'text': clean, 'chunk_id': chunk_id,
                'model': model_id, 'is_final': False, 'language': language,
            })
    except WebSocketDisconnect:
        print('📴 WebSocket client disconnected.')

# ============================================================
# SECTION 6: Intent Predictor (Qwen LLM — GPU only)
# ============================================================
# On CPU: returns 501 so the app falls back to local phrase chips.
# On GPU: lazy-loads Qwen 2.5 1.5B and expands fragmented speech.

llm_model, llm_tokenizer = None, None

def load_llm():
    global llm_model, llm_tokenizer
    if llm_model: return
    print('\n🧠 Loading Qwen 2.5 1.5B Instruct...')
    mid = 'Qwen/Qwen2.5-1.5B-Instruct'
    llm_tokenizer = AutoTokenizer.from_pretrained(mid)
    llm_model     = AutoModelForCausalLM.from_pretrained(mid, torch_dtype=DTYPE, device_map=DEVICE)
    print('✅ Intent Predictor loaded!')

class PredictRequest(BaseModel):
    text: str
    language: str = 'tw'

@app.post('/predict/intent')
async def predict_intent(req: PredictRequest):
    if not LLM_ENABLED:
        # App PredictionService catches this 501 and uses local phrase chips instead
        return JSONResponse(status_code=501,
            content={'error': 'LLM disabled in CPU mode. Using local suggestions.'})

    load_llm()
    lang_name = ('Akan/Twi' if req.language in ['tw', 'twi', 'akan']
                 else 'Ga' if req.language == 'ga' else 'English')
    messages = [
        {'role': 'system', 'content': (
            f'You are a medical AI for a speech-impaired patient. '
            f'Expand their fragmented {lang_name} speech into ONE complete, '
            f'polite sentence in {lang_name}. Output ONLY the sentence.'
        )},
        {'role': 'user', 'content': req.text},
    ]
    text_in = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs  = llm_tokenizer(text_in, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = llm_model.generate(**inputs, max_new_tokens=50, temperature=0.3,
                                  do_sample=True, pad_token_id=llm_tokenizer.eos_token_id)
    predicted = llm_tokenizer.decode(out[0][inputs.input_ids.shape[1]:],
                                      skip_special_tokens=True).strip()
    return {'predicted': predicted, 'language': req.language}

# ============================================================
# SECTION 7: TTS Engine (Meta MMS)
# ============================================================

tts_models = {}

def load_tts(lang_code):
    if lang_code in tts_models: return tts_models[lang_code]
    model_id = 'facebook/mms-tts-aka' if lang_code == 'tw' else 'facebook/mms-tts-eng'
    print(f'\n🌟 Loading TTS ({model_id}) on {DEVICE.upper()}...')
    model     = VitsModel.from_pretrained(model_id).to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tts_models[lang_code] = (model, tokenizer)
    print(f'✅ TTS Loaded: {model_id}')
    return model, tokenizer

@app.post('/tts/synthesize')
@app.get('/tts/synthesize')
async def synthesize(request_or_text, language: str = 'tw'):
    text    = request_or_text.text if hasattr(request_or_text, 'text') else request_or_text
    lang_id = 'tw' if language in ['tw', 'twi', 'akan'] else 'eng'
    model, tokenizer = load_tts(lang_id)
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        output = model(**inputs).waveform
    audio_np = output.squeeze().cpu().numpy()
    sr       = model.config.sampling_rate
    audio_np = np.pad(audio_np, (0, int(0.5 * sr)), mode='constant')  # tail-clip fix
    audio_np = (audio_np * 32767.0).astype(np.int16)
    audio_io = io.BytesIO()
    scipy.io.wavfile.write(audio_io, sr, audio_np)
    audio_io.seek(0)
    return StreamingResponse(audio_io, media_type='audio/wav')

# ============================================================
# SECTION 8: Deployment — Cloudflare Tunnel + Uvicorn
# ============================================================

print('🌍 Spinning up Cloudflare Tunnel...')
subprocess.Popen('nohup ./cloudflared tunnel --url http://127.0.0.1:8000 > cloudflare.log 2>&1 &',
                 shell=True)
time.sleep(10)

with open('cloudflare.log', 'r') as log:
    content = log.read()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
    if match:
        public_url = match.group(0)
        mode_label = '⚡ GPU + LLM' if GPU_AVAILABLE else '🐢 CPU (no LLM)'
        print('\n' + '=' * 70)
        print(f'✅ VOICEAID BACKEND IS ONLINE  [{mode_label}]')
        print(f'👉 COPY THIS URL INTO constants/config.ts:  {public_url}')
        print('=' * 70 + '\n')
    else:
        print('⚠️ Tunnel URL not found — check cloudflare.log')

config = uvicorn.Config(app, host='127.0.0.1', port=8000, ws='websockets')
server = uvicorn.Server(config)
await server.serve()

## API Reference

| Method | Endpoint | GPU | CPU |
|--------|----------|-----|-----|
| `GET` | `/` | ✅ | ✅ |
| `GET` | `/health` | ✅ | ✅ |
| `POST` | `/asr/transcribe` | ✅ Fast | ⚠️ Slow |
| `WebSocket` | `/asr/stream` | ✅ Fast | ⚠️ Slow |
| `POST` | `/predict/intent` | ✅ Qwen LLM | ❌ 501 → app uses local chips |
| `GET/POST` | `/tts/synthesize` | ✅ Fast | ⚠️ Slow |
| `GET` | `/docs` | ✅ | ✅ |

## Notes
- The URL changes every Colab session — always update `constants/config.ts`.
- Models are lazy-loaded on first request. The Qwen LLM only loads on the first `/predict/intent` call.
- On CPU, the app's local medical phrase chips still work perfectly — patients just won't get the AI-expanded sentence, only vocabulary matches.